# Build a Genie + RAG Multi-Agent System

## Workshop Part 3: Take-Home Exercise

Build your own multi-agent orchestrator that routes questions between **Genie** (structured data) and **RAG** (document search).

| Part | What you'll build | Fill-ins |
|------|------------------|----------|
| **Setup** | Imports, config, GenieUtils (pre-solved) | 0 |
| **A** | Explore the RAG infrastructure | 0 |
| **B** | Build a RAG agent function | 2 |
| **C** | Build a keyword + LLM router | 2 |
| **D** | Wire it into an orchestrator | 2 |
| **E** | *(Bonus)* Rebuild with LangGraph | 3 |

### Prerequisites
- Completed **notebook 01** (Genie SDK demo)
- Run **00b_setup_rag** (creates Vector Search index + loads documents)

### Answer Key
The production versions of everything you build here live in `src/agents/`:
- `rag_agent.py` → RAGAgent class
- `supervisor.py` → LangGraph orchestration
- `genie_agent.py` → GenieDataAgent class

In [ ]:
%pip install databricks-sdk databricks-langchain databricks-vectorsearch langgraph langchain-core -q

dbutils.library.restartPython()

In [ ]:
import json
import os
import sys
import time
from datetime import timedelta
from typing import Any

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import GenieMessage
from databricks_langchain import ChatDatabricks
from IPython.display import Markdown, display

# Path setup for Databricks notebooks
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print(f"Python path configured: {sys.path[0]}")

In [ ]:
dbutils.widgets.text("genie_space_id", "", "1. Genie Space ID")
dbutils.widgets.text("catalog", "workshop", "2. Catalog")
dbutils.widgets.text("vs_endpoint", "rag-workshop-endpoint", "3. VS Endpoint")
dbutils.widgets.text("model_endpoint", "databricks-meta-llama-3-3-70b-instruct", "4. Model Endpoint")

In [ ]:
# Read widget values
GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")
CATALOG = dbutils.widgets.get("catalog")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
MODEL_ENDPOINT = dbutils.widgets.get("model_endpoint")

VS_INDEX = f"{CATALOG}.rag.document_index"

# Initialize clients
w = WorkspaceClient()
llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, temperature=0.1)

# Vector Search client + index
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)

print(f"Genie Space:  {GENIE_SPACE_ID}")
print(f"VS Index:     {VS_INDEX}")
print(f"LLM Endpoint: {MODEL_ENDPOINT}")
print(f"All clients initialized ✓")

In [ ]:
# ─────────────────────────────────────────────────
# PROVIDED — You built this in the live demo (notebook 01).
# All TODOs are already filled in so this notebook is self-contained.
# ─────────────────────────────────────────────────

import pandas as pd


class GenieUtils:
    @staticmethod
    def print_output(message: GenieMessage) -> None:
        """Pretty-print a GenieMessage: question, SQL, and answer."""
        sql = None
        answer = None
        for att in message.attachments or []:
            if att.query and att.query.query:
                sql = att.query.query
            if att.text and att.text.content:
                answer = att.text.content

        print(f"Question:\n  {message.content}\n")
        if sql:
            print(f"SQL:\n  {sql}\n")
        if answer:
            print(f"Answer:\n  {answer}")

    @staticmethod
    def ask_genie(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        """Send a question to a Genie Space and return a parsed result dict."""
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = client.genie.start_conversation_and_wait(
            space_id=space_id,
            content=query,
            timeout=timedelta(seconds=120),
        )

        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def to_dataframe(result: dict, client: WorkspaceClient = None) -> "pd.DataFrame":
        """Convert a Genie result dict into a pandas DataFrame."""
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = result["message"]
        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        for att in msg_dict.get("attachments") or []:
            query_att = att.get("query")
            if query_att and query_att.get("statement_id"):
                qr = client.genie.get_message_attachment_query_result(
                    space_id=msg_dict["space_id"],
                    conversation_id=msg_dict["conversation_id"],
                    message_id=msg_dict["id"],
                    attachment_id=att["id"],
                )

                stmt = qr.statement_response
                columns = [col.name for col in stmt.manifest.schema.columns]
                rows = stmt.result.data_array
                return pd.DataFrame(rows, columns=columns)

        return pd.DataFrame([{"answer": result.get("text", "")}])

    @staticmethod
    def ask_genie_verbose(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        """Query Genie with verbose status output (polling loop)."""
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        conv = client.genie.start_conversation(
            space_id=space_id,
            content=query,
        )
        conversation_id = conv.conversation_id
        message_id = conv.message_id
        print(f"[verbose] Conversation started: {conversation_id}")
        print(f"[verbose] Message ID: {message_id}")

        last_status = None
        while True:
            msg = client.genie.get_message(
                space_id=space_id,
                conversation_id=conversation_id,
                message_id=message_id,
            )
            current_status = msg.status.value if msg.status else "UNKNOWN"
            if current_status != last_status:
                print(f"[verbose] Status: {current_status}")
                last_status = current_status

            if current_status in ("COMPLETED", "FAILED"):
                break
            time.sleep(2)

        if current_status == "FAILED":
            error = msg.error if hasattr(msg, "error") else "Unknown error"
            print(f"[verbose] Genie returned an error: {error}")
            return {"message": msg, "sql": None, "description": None, "data": None, "text": None}

        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)
        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None
        print(f"[verbose] Done — SQL: {'yes' if sql else 'no'}, Text: {'yes' if text else 'no'}")

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def get_space_metadata(space_id: str, client: WorkspaceClient = None) -> dict:
        """Fetch metadata for a Genie Space."""
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        space = client.genie.get_space(
            space_id=space_id,
            include_serialized_space=True,
        )

        serialized = space.serialized_space
        if not serialized:
            raise RuntimeError("serialized_space missing in response")

        cfg = json.loads(serialized)
        config = cfg.get("config", {})
        data_sources = cfg.get("data_sources", {})

        sample_questions = [
            " ".join(q.get("question", []))
            for q in config.get("sample_questions", [])
        ]

        tables = []
        for t in data_sources.get("tables", []):
            tables.append({
                "identifier": t.get("identifier"),
                "description": t.get("description"),
                "columns": [
                    {
                        "column_name": col_meta.get("column_name"),
                        "description": col_meta.get("description"),
                    }
                    for col_meta in t.get("column_configs", [])
                ],
            })

        return {
            "space_id": space.space_id,
            "title": space.title,
            "description": space.description,
            "sample_questions": sample_questions,
            "tables": tables,
        }


print("GenieUtils loaded (all methods pre-solved) ✓")

## Part A — Explore the RAG Infrastructure

Notebook `00b_setup_rag` created everything you need for document search:

| Component | What it is |
|-----------|----------|
| **Delta Table** | `{catalog}.rag.document_chunks` — chunked markdown documents |
| **VS Endpoint** | Hosts the Vector Search index |
| **VS Index** | `{catalog}.rag.document_index` — embeddings for similarity search |
| **Documents** | Company policies: warranty, service, returns, financing, safety, compensation |

Let's verify the infrastructure is ready and see how similarity search works.

In [ ]:
# Verify the document_chunks table
full_table = f"{CATALOG}.rag.document_chunks"

count = spark.sql(f"SELECT COUNT(*) FROM {full_table}").first()[0]
print(f"Total document chunks: {count}\n")

# Source breakdown
sources = spark.sql(f"""
    SELECT source, COUNT(*) as chunks 
    FROM {full_table} 
    GROUP BY source 
    ORDER BY source
""").collect()

print("Documents loaded:")
for row in sources:
    print(f"  {row.source:40s} ({row.chunks} chunks)")

In [ ]:
# Test similarity search directly on the index
test_query = "What is the vehicle warranty policy?"

results = index.similarity_search(
    query_text=test_query,
    columns=["id", "content", "source", "metadata"],
    num_results=3,
)

print(f"Query: {test_query}")
print(f"Results: {results['result']['row_count']} matches\n")

for i, row in enumerate(results["result"]["data_array"], 1):
    # Row format: [id, content, source, metadata, score]
    print(f"--- Result {i} (score: {row[4]:.4f}) ---")
    print(f"Source: {row[2]}")
    print(f"Content: {row[1][:200]}...")
    print()

### Recap

`index.similarity_search()` returns a dict with this structure:
```
{"result": {"data_array": [[id, content, source, metadata, score], ...], "row_count": N}}
```

Each row in `data_array` contains the columns you requested plus a relevance **score** (last element).
Higher scores = more relevant to your query.

## Part B — Build a RAG Agent Function

Now build `ask_rag()` — a function that:
1. Queries Vector Search for relevant document chunks
2. Builds a prompt with the retrieved context
3. Sends the prompt to the LLM for a grounded answer

```
Question → Vector Search → Top-K chunks → LLM prompt → Answer
```

**2 TODOs** in the next cell.

In [ ]:
def ask_rag(question: str, num_results: int = 3) -> str:
    """Query documents via Vector Search and generate an LLM answer.
    
    Args:
        question: Natural language question about company documents/policies
        num_results: Number of document chunks to retrieve
        
    Returns:
        LLM-generated answer grounded in the retrieved documents
    """
    
    # ── TODO 1 ─────────────────────────────────────
    # Call index.similarity_search() to find relevant document chunks.
    #
    # Hint: index.similarity_search(
    #     query_text=???,
    #     columns=["id", "content", "source", "metadata"],
    #     num_results=???
    # )
    #
    # Answer key: src/agents/rag_agent.py → RAGAgent._real_query()
    # ──────────────────────────────────────────────
    results = ...  # TODO 1
    
    # Parse the results
    data_array = results.get("result", {}).get("data_array", [])
    
    # Build context from retrieved chunks
    context_parts = []
    sources = []
    for i, row in enumerate(data_array, 1):
        content = row[1]
        source = row[2]
        score = row[4]
        context_parts.append(f"[Document {i}: {source} (score: {score:.2f})]\n{content}")
        sources.append(f"  {i}. {source} (relevance: {score:.2f})")
    
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are a helpful assistant answering questions based on company documents.

CONTEXT (Retrieved Documents):
{context}

USER QUESTION: {question}

INSTRUCTIONS:
- Answer based ONLY on the provided context
- If the context doesn't fully answer the question, say what's missing
- Be concise but thorough
- Mention which document the information came from

ANSWER:"""
    
    # ── TODO 2 ─────────────────────────────────────
    # Call the LLM with the prompt and return the answer.
    #
    # Hint: response = llm.invoke(prompt)
    #       return response.content
    #
    # Answer key: src/agents/rag_agent.py → RAGAgent._generate_answer()
    # ──────────────────────────────────────────────
    response = ...  # TODO 2
    
    answer = response.content
    print(f"Sources:\n" + "\n".join(sources))
    return answer

In [ ]:
# Test ask_rag with different document topics
print("=" * 60)
print("Test 1: Warranty question")
print("=" * 60)
answer1 = ask_rag("What does the vehicle warranty cover?")
print(f"\nAnswer:\n{answer1}\n")

print("=" * 60)
print("Test 2: Service question")
print("=" * 60)
answer2 = ask_rag("When is the first service required?")
print(f"\nAnswer:\n{answer2}")

### Recap

You now have **two agent functions** that answer different types of questions:

| Function | Data source | Best for |
|----------|------------|----------|
| `GenieUtils.ask_genie()` | Genie Space (SQL) | Metrics, trends, aggregations — anything in your tables |
| `ask_rag()` | Vector Search (documents) | Policies, procedures, guidelines — unstructured knowledge |

**Next:** Build a router that decides which function to call.

## Part C — Build a Router

The router examines each question and decides: **Genie** (data) or **RAG** (documents)?

```
Question
   |
   v
+----------+
|  Router  |---- "rag"  --> ask_rag()
|          |---- "genie" --> GenieUtils.ask_genie()
+----------+
```

You'll build two versions:
1. **Keyword router** — fast, deterministic, no LLM cost
2. **LLM router** — smarter, handles ambiguous questions

**2 TODOs** across the next two cells.

In [ ]:
DOC_KEYWORDS = [
    "policy", "document", "procedure", "guide", "how to",
    "warranty", "return", "exchange", "financing", "service",
    "maintenance", "safety", "compliance", "compensation",
]

def route_keyword(question: str) -> str:
    """Route a question using keyword matching.
    
    Args:
        question: User's question
        
    Returns:
        "rag" if the question matches document keywords, "genie" otherwise
    """
    # ── TODO 3 ─────────────────────────────────────
    # Loop over DOC_KEYWORDS. If any keyword appears in the
    # lowercased question, return "rag". Otherwise return "genie".
    #
    # Hint:
    #   q_lower = question.lower()
    #   for kw in DOC_KEYWORDS:
    #       if kw in q_lower:
    #           return "rag"
    #   return "genie"
    #
    # Answer key: src/agents/supervisor.py → DOC_KEYWORDS + mock_route_query()
    # ──────────────────────────────────────────────
    pass  # TODO 3


# ── Test harness ──
test_questions = [
    ("What is the vehicle warranty coverage?", "rag"),
    ("What are total sales by region?", "genie"),
    ("Explain the return policy", "rag"),
    ("Top 5 selling vehicle models?", "genie"),
    ("What is the service maintenance schedule?", "rag"),
    ("How many customers bought EVs last quarter?", "genie"),
]

print("Keyword Router Tests:")
print("=" * 60)
for question, expected in test_questions:
    result = route_keyword(question)
    status = "✓" if result == expected else "✗"
    print(f"  {status} {result:5s} ← {question}")

In [ ]:
def route_llm(question: str) -> str:
    """Route a question using the LLM.
    
    Args:
        question: User's question
        
    Returns:
        "rag" or "genie"
    """
    prompt = f"""You are a query router. Given a user question, decide which agent should handle it.

AGENTS:
- "genie": For questions about structured data — sales, revenue, metrics, trends, counts, 
  aggregations, or anything that would be answered with SQL against a database.
- "rag": For questions about company documents, policies, procedures, guidelines, 
  warranties, compliance, or any unstructured knowledge.

Return ONLY a JSON object: {{"route": "genie"}} or {{"route": "rag"}}

Question: {question}"""
    
    # ── TODO 4 ─────────────────────────────────────
    # Call the LLM with the prompt, parse the JSON response,
    # and return the "route" value.
    #
    # Hint:
    #   response = llm.invoke(prompt)
    #   return json.loads(response.content)["route"]
    #
    # Answer key: See ask_router() in notebook 01
    # ──────────────────────────────────────────────
    pass  # TODO 4


# Test with the same questions
print("LLM Router Tests:")
print("=" * 60)
for question, expected in test_questions:
    result = route_llm(question)
    status = "✓" if result == expected else "~"  # LLM may disagree on edge cases
    print(f"  {status} {result:5s} ← {question}")

### Recap: Keyword vs LLM Routing

| | Keyword Router | LLM Router |
|---|---|---|
| **Speed** | Instant | ~1 second (LLM call) |
| **Cost** | Free | 1 LLM call per question |
| **Accuracy** | Good for clear-cut questions | Better for ambiguous questions |
| **Maintenance** | Must update keyword list manually | Adapts to new question patterns |

**Rule of thumb:** Start with keyword routing. Add LLM routing when you need to handle ambiguous or edge-case questions.

## Part D — Wire It Into an Orchestrator

Combine everything into a `GenieRAGOrchestrator` class:

```
Question
   |
   v
+------------+
|   Router   | (keyword or LLM)
+-----+------+
      |
  +---+---+
  |       |
  v       v
Genie    RAG
  |       |
  +---+---+
      |
      v
   Answer
```

**2 TODOs** in the next cell.

In [ ]:
class GenieRAGOrchestrator:
    """Orchestrator that routes questions to Genie (data) or RAG (documents)."""
    
    def __init__(self, space_id: str, router: str = "keyword"):
        """
        Args:
            space_id: Genie Space ID for data queries
            router: "keyword" or "llm"
        """
        self.space_id = space_id
        self.router = router
    
    def execute(self, question: str) -> str:
        """Route and execute a question.
        
        Args:
            question: User's question
            
        Returns:
            Answer string from either Genie or RAG
        """
        # Step 1: Route the question
        if self.router == "llm":
            route = route_llm(question)
        else:
            route = route_keyword(question)
        
        print(f"[Router: {self.router}] → {route}")
        
        # Step 2: Execute based on route
        if route == "rag":
            # ── TODO 5 ─────────────────────────────────────
            # Call ask_rag() with the question and return the result.
            #
            # Hint: result = ask_rag(question)
            #
            # Answer key: src/agents/rag_agent.py → RAGAgent.query()
            # ──────────────────────────────────────────────
            result = ...  # TODO 5
            return result
        else:
            # ── TODO 6 ─────────────────────────────────────
            # Call GenieUtils.ask_genie() with the question, space_id, and client.
            # Return the text answer from the result dict.
            #
            # Hint: genie_result = GenieUtils.ask_genie(question, self.space_id, client=w)
            #       return genie_result.get("text", "No answer returned.")
            #
            # Answer key: src/agents/genie_agent.py → GenieDataAgent.query()
            # ──────────────────────────────────────────────
            genie_result = ...  # TODO 6
            return genie_result

In [ ]:
orch = GenieRAGOrchestrator(space_id=GENIE_SPACE_ID, router="keyword")

test_questions_mixed = [
    "What does the vehicle warranty cover?",
    "What are the top 5 selling models by revenue?",
    "Explain the return and exchange policy",
    "How many orders were placed last month?",
    "What is the service maintenance schedule?",
    "Which region has the highest sales?",
]

print("Orchestrator Test (keyword router)")
print("=" * 60)
for q in test_questions_mixed:
    print(f"\nQ: {q}")
    answer = orch.execute(q)
    preview = answer[:300] + "..." if len(answer) > 300 else answer
    print(f"A: {preview}")
    print("-" * 60)

In [ ]:
orch_llm = GenieRAGOrchestrator(space_id=GENIE_SPACE_ID, router="llm")

# Ambiguous questions that benefit from LLM routing
ambiguous_questions = [
    "What safety inspections are required before selling a vehicle?",
    "Compare the financing options available to fleet customers",
    "What's the process for handling a warranty claim?",
]

print("Orchestrator Test (LLM router)")
print("=" * 60)
for q in ambiguous_questions:
    print(f"\nQ: {q}")
    answer = orch_llm.execute(q)
    preview = answer[:300] + "..." if len(answer) > 300 else answer
    print(f"A: {preview}")
    print("-" * 60)

### Recap

You now have a working **Genie + RAG orchestrator**:
- Routes questions to the right agent (Genie for data, RAG for documents)
- Supports both keyword-based and LLM-based routing
- Returns answers from structured data OR unstructured documents

**Limitations of this approach:**
- Simple if/else routing — no tool-calling, no conversation memory
- Single-turn only — no follow-up questions
- No parallel execution or multi-step reasoning

**Next:** Part E rebuilds this as a proper LangGraph agent with tool-calling and a supervisor pattern.

## Part E — *(Bonus)* Rebuild with LangGraph

LangGraph lets you build agents as **stateful graphs** with nodes and edges.

| Concept | Description |
|---------|-------------|
| **State** | TypedDict that flows through the graph (messages, results, etc.) |
| **Node** | A function that reads state and returns updates |
| **Edge** | Connection between nodes (can be conditional) |
| **Tool** | A `@tool`-decorated function the LLM can call |

```
         START
           |
           v
     +-----------+
     | Supervisor|<--------------+
     |  (LLM)    |               |
     +-----+-----+               |
           |                     |
     +-----+-----+               |
     |           |               |
     v           v               |
   Tools       END               |
     |                           |
     v                           |
  after_tools -------------------+
```

**3 TODOs** across the next two cells.

In [ ]:
import operator
from collections.abc import Sequence
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import ToolNode


# State definition
class AgentState(TypedDict):
    """State that flows through the LangGraph agent."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next_agent: str


# ── TODO 7 ─────────────────────────────────────
# Create two @tool functions:
#
# 1. query_data(question: str) -> str
#    - Call GenieUtils.ask_genie(question, GENIE_SPACE_ID, client=w)
#    - Return a formatted string with the SQL and text answer
#
# 2. search_documents(question: str) -> str
#    - Call ask_rag(question)
#    - Return the answer string
#
# Hint — both follow this pattern:
#   @tool
#   def my_tool(question: str) -> str:
#       """Docstring the LLM reads to decide when to call this tool."""
#       result = ...
#       return formatted_result
#
# Answer key: src/agents/supervisor.py → create_agent_tools()
# ──────────────────────────────────────────────

# @tool
# def query_data(question: str) -> str:
#     """Query structured data using natural language. Use this for questions about
#     metrics, sales, revenue, products, customers, trends, and any data analysis."""
#     result = GenieUtils.ask_genie(question, GENIE_SPACE_ID, client=w)
#     parts = []
#     if result.get("sql"):
#         parts.append(f"**SQL:**\n```sql\n{result['sql']}\n```")
#     if result.get("text"):
#         parts.append(f"**Answer:** {result['text']}")
#     return "\n\n".join(parts) if parts else "No results returned."

# @tool
# def search_documents(question: str) -> str:
#     """Search company documents and policies. Use this for questions about
#     policies, procedures, documentation, guides, warranties, compliance,
#     and other non-data questions."""
#     return ask_rag(question)


# Verify tools are defined (uncomment after implementing)
# tools = [query_data, search_documents]
# print(f"Tools registered: {[t.name for t in tools]}")

In [ ]:
# ── Pre-filled: supervisor node, after_tools node, route function ──

def supervisor_node(state: AgentState) -> dict:
    """The supervisor uses the LLM to decide which tool to call."""
    messages = state["messages"]
    
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(messages)
    
    if response.tool_calls:
        return {"messages": [response], "next_agent": "tools"}
    return {"messages": [response], "next_agent": "end"}


def after_tools(state: AgentState) -> dict:
    """After tool execution, route back to supervisor."""
    return {"next_agent": "supervisor"}


def route_next(state: AgentState) -> Literal["tools", "supervisor", "__end__"]:
    """Conditional routing based on state."""
    next_agent = state.get("next_agent", "end")
    if next_agent == "tools":
        return "tools"
    elif next_agent == "supervisor":
        return "supervisor"
    return "__end__"


tool_node = ToolNode(tools)


# ── TODO 8 ─────────────────────────────────────
# Build the StateGraph:
#   1. Create: workflow = StateGraph(AgentState)
#   2. Add nodes: "supervisor", "tools" (tool_node), "after_tools"
#   3. Set entry point: "supervisor"
#
# Hint:
#   workflow = StateGraph(AgentState)
#   workflow.add_node("supervisor", supervisor_node)
#   workflow.add_node("tools", tool_node)
#   workflow.add_node("after_tools", after_tools)
#   workflow.set_entry_point("supervisor")
#
# Answer key: src/agents/supervisor.py → create_supervisor_agent()
# ──────────────────────────────────────────────
workflow = ...  # TODO 8


# ── TODO 9 ─────────────────────────────────────
# Add edges:
#   1. Conditional edges from "supervisor" using route_next
#   2. Edge from "tools" to "after_tools"
#   3. Edge from "after_tools" to "supervisor"
#   4. Compile the graph
#
# Hint:
#   workflow.add_conditional_edges("supervisor", route_next, 
#       {"tools": "tools", "supervisor": "supervisor", "__end__": END})
#   workflow.add_edge("tools", "after_tools")
#   workflow.add_edge("after_tools", "supervisor")
#   graph = workflow.compile()
#
# Answer key: src/agents/supervisor.py → create_supervisor_agent()
# ──────────────────────────────────────────────
# graph = ...  # TODO 9

# Verify (uncomment after implementing)
# print(f"Graph compiled with {len(graph.nodes)} nodes")

In [ ]:
def ask_agent(question: str) -> str:
    """Convenience wrapper to invoke the LangGraph agent."""
    result = graph.invoke({
        "messages": [HumanMessage(content=question)],
        "next_agent": "supervisor",
    })
    
    # Extract the final AI response
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and not msg.tool_calls:
            return msg.content
    return "No response generated."


# Test with mixed questions
test_questions_lg = [
    "What does the vehicle warranty cover?",
    "What are the top 5 selling models?",
    "Explain the return policy",
    "How many customers are in each segment?",
]

print("LangGraph Agent Tests")
print("=" * 60)
for q in test_questions_lg:
    print(f"\nQ: {q}")
    answer = ask_agent(q)
    preview = answer[:400] + "..." if len(answer) > 400 else answer
    print(f"A: {preview}")
    print("-" * 60)

## Summary

### What You Built

| Part | Component | Pattern |
|------|-----------|--------|
| **B** | `ask_rag()` | Vector Search → LLM prompt → grounded answer |
| **C** | `route_keyword()` / `route_llm()` | Keyword matching vs LLM-based routing |
| **D** | `GenieRAGOrchestrator` | Router → Genie or RAG → answer |
| **E** | LangGraph agent | Supervisor → Tool nodes → stateful graph |

### Architecture Progression

```
Part D (Simple)                    Part E (LangGraph)
-----------------                  ------------------
Question                           Question
   |                                  |
   v                                  v
 Router --> if/else              Supervisor (LLM)
   |                                  |
   +-> Genie                     +----+----+
   +-> RAG                      Tool      END
                                 Node
                                  |
                                  +-> query_data (Genie)
                                  +-> search_documents (RAG)
```

### Answer Key

The production versions live in `src/agents/`:
- **`rag_agent.py`** → `RAGAgent` class with error handling + mock mode
- **`supervisor.py`** → Full LangGraph orchestration with `AgentState`, tools, and routing
- **`genie_agent.py`** → `GenieDataAgent` with caching, retry, and conversation support

### Next Steps
- Try **notebook 02** for multi-Genie orchestration with cross-domain synthesis
- Explore `src/agents/multi_genie_orchestrator.py` for fan-out/synthesize patterns
- Add memory to the LangGraph agent with `MemorySaver`

In [ ]:
print("Notebook 03 complete!")
print()
print("Key takeaways:")
print("  1. RAG adds document search alongside Genie's SQL capabilities")
print("  2. Routing decides which agent handles each question")
print("  3. An orchestrator ties routing + agents into a single interface")
print("  4. LangGraph provides a production-grade framework for multi-agent systems")
print()
print("Answer key:")
print(f"  src/agents/rag_agent.py      — RAGAgent class")
print(f"  src/agents/supervisor.py     — LangGraph supervisor + tools")
print(f"  src/agents/genie_agent.py    — GenieDataAgent class")